<a href="https://colab.research.google.com/github/tustus1022-ui/esaa/blob/main/2%EC%A3%BC%EC%B0%A8_lightgbm%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

전처리2 (500): selection + PCA(500) + lag feature

In [ ]:
# 필요한 패키지 설치
!pip install -q numerapi lightgbm scikit-learn pandas pyarrow

In [ ]:
# 라이브러리 불러오기
import json
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from numerapi import NumerAPI
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
import lightgbm as lgb

napi = NumerAPI()
VERSION = "v5.2"

In [ ]:
# 데이터 다운로드 + 기본 설정
# Colab-Numerai 서버 간 네트워크가 대용량 파일(특히 validation.parquet
# 4.31GB) 다운로드 도중 끊기는 경우가 있어서, 실패하면 자동으로
# 재시도하도록 합니다.
# -----------------------------------------------------------------
def download_with_retry(filename, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            napi.download_dataset(filename)
            return
        except Exception as e:
            print(f"[재시도 {attempt}/{max_retries}] {filename} 다운로드 실패: {e}")
            if attempt == max_retries:
                raise
    return

download_with_retry(f"{VERSION}/features.json")
download_with_retry(f"{VERSION}/train.parquet")
download_with_retry(f"{VERSION}/validation.parquet")

feature_metadata = json.load(open(f"{VERSION}/features.json"))
all_feature_cols = feature_metadata["feature_sets"]["all"]

TARGET_COL = "target_cyrusd_20"   # 서윤님과 동일한 메인 타겟
PCA_N = 500                        # PCA component 개수 (시현님 담당 = 500)
LAGS = (1, 2)                      # era 단위 lag
FIT_BATCH_SIZE = 20_000            # 전체 데이터 스트리밍(selection/scaler/PCA fit)용 청크 크기
FINAL_MAX_ROWS_PER_ERA = 400       # 최종 학습 데이터를 만들 때 era당 최대 row 수
RANDOM_STATE = 42

print("전체 feature 수:", len(all_feature_cols))

전체 feature 수: 2748


In [ ]:
# 공용 스트리밍 유틸 함수
def iter_parquet_batches(path, columns, batch_size):
    """parquet 파일을 컬럼 제한 + 배치 단위로 스트리밍해서 순회.
    파일 전체를 한 번에 메모리에 올리지 않기 때문에 파일 크기와 무관하게 안전합니다."""
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        chunk = batch.to_pandas()
        chunk["era"] = chunk["era"].astype(int)
        yield chunk

def clean_array(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
# Feature Selection - 전체 데이터 스트리밍 상관계수 (샘플링 없음)
# -----------------------------------------------------------------
# 팀원분 코드와 같은 방식: 누적합(sum_x, sum_x2, sum_xy ...)을 청크마다
# 쌓아가면서 최종적으로 전체 데이터 기준 상관계수를 계산합니다.
# LightGBM importance 대신 상관계수를 쓰는 이유: 상관계수는 이렇게
# 누적합만으로 정확히 계산 가능해서 전체 데이터를 다 쓸 수 있지만,
# LightGBM importance는 전체 데이터를 한 번에 메모리에 올려야만 계산 가능합니다.
# -----------------------------------------------------------------
selection_columns = ["era"] + all_feature_cols + [TARGET_COL]
p = len(all_feature_cols)

sum_x = np.zeros(p, dtype=np.float64)
sum_x2 = np.zeros(p, dtype=np.float64)
sum_xy = np.zeros(p, dtype=np.float64)
sum_y = 0.0
sum_y2 = 0.0
n_total = 0

for chunk in iter_parquet_batches(f"{VERSION}/train.parquet", selection_columns, FIT_BATCH_SIZE):
    X_chunk = clean_array(chunk[all_feature_cols].to_numpy(dtype=np.float32))
    y_chunk = clean_array(chunk[TARGET_COL].to_numpy(dtype=np.float32))

    n_chunk = len(y_chunk)
    n_total += n_chunk

    sum_x += X_chunk.sum(axis=0, dtype=np.float64)
    sum_x2 += (X_chunk * X_chunk).sum(axis=0, dtype=np.float64)
    sum_xy += X_chunk.T @ y_chunk.astype(np.float64)
    sum_y += y_chunk.sum(dtype=np.float64)
    sum_y2 += (y_chunk * y_chunk).sum(dtype=np.float64)

    print(f"Feature selection 진행: 누적 {n_total} rows")

cov_xy = sum_xy - (sum_x * sum_y / n_total)
var_x = sum_x2 - (sum_x ** 2 / n_total)
var_y = sum_y2 - (sum_y ** 2 / n_total)

corr = cov_xy / (np.sqrt(var_x * var_y) + 1e-12)
corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
scores = np.abs(corr)

threshold = np.median(scores)  # 서윤님과 동일하게 "상위 50%" 컨셉 유지
selected_features = [f for f, s in zip(all_feature_cols, scores) if s >= threshold]

print(f"전체 rows 사용: {n_total}")
print(f"전체 feature 수: {len(all_feature_cols)}")
print(f"선택된 feature 수: {len(selected_features)}")

Feature selection 진행: 누적 20000 rows
Feature selection 진행: 누적 40000 rows
Feature selection 진행: 누적 60000 rows
Feature selection 진행: 누적 80000 rows
Feature selection 진행: 누적 100000 rows
Feature selection 진행: 누적 120000 rows
Feature selection 진행: 누적 140000 rows
Feature selection 진행: 누적 160000 rows
Feature selection 진행: 누적 180000 rows
Feature selection 진행: 누적 200000 rows
Feature selection 진행: 누적 220000 rows
Feature selection 진행: 누적 240000 rows
Feature selection 진행: 누적 260000 rows
Feature selection 진행: 누적 280000 rows
Feature selection 진행: 누적 300000 rows
Feature selection 진행: 누적 320000 rows
Feature selection 진행: 누적 340000 rows
Feature selection 진행: 누적 360000 rows
Feature selection 진행: 누적 380000 rows
Feature selection 진행: 누적 400000 rows
Feature selection 진행: 누적 420000 rows
Feature selection 진행: 누적 440000 rows
Feature selection 진행: 누적 460000 rows
Feature selection 진행: 누적 480000 rows
Feature selection 진행: 누적 500000 rows
Feature selection 진행: 누적 520000 rows
Feature selection 진행: 누적 540000 rows
Featu

In [ ]:
# Scaling + PCA 학습 - 전체 데이터 스트리밍 partial_fit (샘플링 없음)
# -----------------------------------------------------------------
# StandardScaler와 IncrementalPCA는 partial_fit을 지원해서,
# 전체 데이터를 청크로 여러 번 나눠 보여줘도 "전체 데이터 기준"으로
# 정확하게 학습됩니다 (한 번에 다 메모리에 올릴 필요 없음).
# 진행 상황을 볼 수 있도록 print를 추가함
# -----------------------------------------------------------------
fit_columns = ["era"] + selected_features
PCA_FIT_BATCH_SIZE = 50_000  # 속도를 위해 selection 때보다 batch를 키움

scaler = StandardScaler()
n_seen = 0
for chunk in iter_parquet_batches(f"{VERSION}/train.parquet", fit_columns, PCA_FIT_BATCH_SIZE):
    X_chunk = clean_array(chunk[selected_features].to_numpy(dtype=np.float32))
    scaler.partial_fit(X_chunk)
    n_seen += len(chunk)
    print(f"[Scaler] 누적 {n_seen} rows 처리")
print("Scaler partial_fit 완료, 총 rows:", n_seen)

PCA_N = min(PCA_N, len(selected_features))
pca = IncrementalPCA(n_components=PCA_N, batch_size=PCA_FIT_BATCH_SIZE)

n_seen = 0
for chunk in iter_parquet_batches(f"{VERSION}/train.parquet", fit_columns, PCA_FIT_BATCH_SIZE):
    X_chunk = clean_array(chunk[selected_features].to_numpy(dtype=np.float32))
    X_scaled = scaler.transform(X_chunk)
    if X_scaled.shape[0] >= PCA_N:  # IncrementalPCA는 배치 크기가 n_components 이상이어야 함
        pca.partial_fit(X_scaled)
    n_seen += len(chunk)
    print(f"[PCA] 누적 {n_seen} rows 처리")

print("PCA component 개수:", PCA_N)
print("PCA 설명 분산 비율 합:", pca.explained_variance_ratio_.sum())

[Scaler] 누적 50000 rows 처리
[Scaler] 누적 100000 rows 처리
[Scaler] 누적 150000 rows 처리
[Scaler] 누적 200000 rows 처리
[Scaler] 누적 250000 rows 처리
[Scaler] 누적 300000 rows 처리
[Scaler] 누적 350000 rows 처리
[Scaler] 누적 400000 rows 처리
[Scaler] 누적 450000 rows 처리
[Scaler] 누적 500000 rows 처리
[Scaler] 누적 550000 rows 처리
[Scaler] 누적 600000 rows 처리
[Scaler] 누적 650000 rows 처리
[Scaler] 누적 700000 rows 처리
[Scaler] 누적 750000 rows 처리
[Scaler] 누적 800000 rows 처리
[Scaler] 누적 850000 rows 처리
[Scaler] 누적 900000 rows 처리
[Scaler] 누적 950000 rows 처리
[Scaler] 누적 1000000 rows 처리
[Scaler] 누적 1050000 rows 처리
[Scaler] 누적 1100000 rows 처리
[Scaler] 누적 1150000 rows 처리
[Scaler] 누적 1200000 rows 처리
[Scaler] 누적 1250000 rows 처리
[Scaler] 누적 1300000 rows 처리
[Scaler] 누적 1350000 rows 처리
[Scaler] 누적 1400000 rows 처리
[Scaler] 누적 1450000 rows 처리
[Scaler] 누적 1500000 rows 처리
[Scaler] 누적 1550000 rows 처리
[Scaler] 누적 1600000 rows 처리
[Scaler] 누적 1650000 rows 처리
[Scaler] 누적 1700000 rows 처리
[Scaler] 누적 1750000 rows 처리
[Scaler] 누적 1800000 rows 처리
[Scaler] 누적 

In [ ]:
# 최종 학습용 데이터 만들기 (era당 row 수 제한 - 여기서만 샘플링)
# -----------------------------------------------------------------
# selection/scaling/PCA는 전체 데이터로 정확히 학습했지만,
# PCA(500) + lag(1000) = 1500개 컬럼짜리 최종 데이터를 "전체 row"로
# 만들면 다시 RAM이 터집니다. 그래서 최종 학습 데이터만 era당 row 수를
# 제한해서 만듭니다 (feature 품질은 이미 전체 데이터 기준으로 확보된 상태).
# -----------------------------------------------------------------
def build_final_dataset(path, era_step, max_rows_per_era, seed=RANDOM_STATE):
    reservoirs = {}
    columns = ["era"] + selected_features + [TARGET_COL]

    for chunk in iter_parquet_batches(path, columns, FIT_BATCH_SIZE):
        for era, grp in chunk.groupby("era"):
            if era_step > 1 and (era % era_step != 0):
                continue
            grp = grp.reset_index(drop=True)
            if era not in reservoirs:
                reservoirs[era] = grp.iloc[:max_rows_per_era].copy()
            else:
                combined = pd.concat([reservoirs[era], grp], ignore_index=True)
                if len(combined) > max_rows_per_era:
                    combined = combined.sample(n=max_rows_per_era, random_state=seed)
                reservoirs[era] = combined

    raw = pd.concat(reservoirs.values(), ignore_index=True)

    X = clean_array(raw[selected_features].to_numpy(dtype=np.float32))
    X_scaled = scaler.transform(X)
    X_pca = pca.transform(X_scaled).astype(np.float32)

    pca_cols = [f"pca_{i}" for i in range(PCA_N)]
    out = pd.DataFrame(X_pca, columns=pca_cols)
    out["era"] = raw["era"].values
    out["target"] = raw[TARGET_COL].astype(np.float32).values
    return out, pca_cols


train_pca_df, pca_cols = build_final_dataset(
    f"{VERSION}/train.parquet", era_step=1, max_rows_per_era=FINAL_MAX_ROWS_PER_ERA
)
val_pca_df, _ = build_final_dataset(
    f"{VERSION}/validation.parquet", era_step=1, max_rows_per_era=FINAL_MAX_ROWS_PER_ERA
)

print("train_pca_df shape:", train_pca_df.shape)
print("val_pca_df shape:", val_pca_df.shape)

train_pca_df shape: (229600, 502)
val_pca_df shape: (260800, 502)


In [ ]:
# Lag feature 추가 (era 단위)
# -----------------------------------------------------------------
# Numerai의 `id`는 종목-era 조합마다 고유해서 개별 종목을 시간에 따라
# 추적할 수 없습니다. 그래서 "종목별 lag"이 아니라 era 단위(시장 전체) 통계의 lag를 만들기.
# -----------------------------------------------------------------
def add_era_level_lag_features(df, pca_cols, lags=LAGS):
    era_mean = df.groupby("era")[pca_cols].mean().sort_index()

    lag_frames = []
    for lag in lags:
        shifted = era_mean.shift(lag)
        shifted.columns = [f"{c}_lag{lag}" for c in pca_cols]
        lag_frames.append(shifted)

    era_lag_features = pd.concat(lag_frames, axis=1)
    df = df.merge(era_lag_features, on="era", how="left")
    return df

train_final = add_era_level_lag_features(train_pca_df, pca_cols)
val_final = add_era_level_lag_features(val_pca_df, pca_cols)

lag_cols = [c for c in train_final.columns if "_lag" in c]
train_final[lag_cols] = train_final[lag_cols].fillna(0)
val_final[lag_cols] = val_final[lag_cols].fillna(0)

print("최종 feature 개수 (PCA + lag):", len(pca_cols) + len(lag_cols))
print("train_final shape:", train_final.shape)

# -----------------------------------------------------------------
# [캐싱] 여기까지 오는 데 시간이 오래 걸렸으니(전체 데이터 4번 스트리밍),
# 지금 상태를 디스크에 저장해둡니다. 이후 모델 학습 단계에서 세션이
# 또 죽더라도, 처음부터 다시 돌리지 않고 아래 코드로 바로 불러올 수 있습니다.
#
#   train_final = pd.read_parquet("train_final_cache.parquet")
#   val_final = pd.read_parquet("val_final_cache.parquet")
# -----------------------------------------------------------------
train_final.to_parquet("train_final_cache.parquet")
val_final.to_parquet("val_final_cache.parquet")
print("캐시 저장 완료: train_final_cache.parquet / val_final_cache.parquet")

최종 feature 개수 (PCA + lag): 1500
train_final shape: (229600, 1502)
캐시 저장 완료: train_final_cache.parquet / val_final_cache.parquet


In [ ]:
import pandas as pd

try:
    train_final = pd.read_parquet("train_final_cache.parquet")
    val_final = pd.read_parquet("val_final_cache.parquet")
    print("캐시 로드 성공! Cell 1~8 다시 안 돌려도 됩니다.")
    print("train_final shape:", train_final.shape)
except FileNotFoundError:
    print("캐시 파일이 없습니다. Cell 1~8을 처음부터 다시 실행해야 합니다.")

캐시 로드 성공! Cell 1~8 다시 안 돌려도 됩니다.
train_final shape: (229600, 1502)


In [ ]:
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

pca_cols = [c for c in train_final.columns if c.startswith("pca_") and "_lag" not in c]
lag_cols = [c for c in train_final.columns if "_lag" in c]

print("pca_cols 개수:", len(pca_cols))
print("lag_cols 개수:", len(lag_cols))

pca_cols 개수: 500
lag_cols 개수: 1000


In [ ]:
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

try:
    train_final = pd.read_parquet("train_final_cache.parquet")
    val_final = pd.read_parquet("val_final_cache.parquet")
    print("캐시 로드 성공! train_final shape:", train_final.shape)

    pca_cols = [c for c in train_final.columns if c.startswith("pca_") and "_lag" not in c]
    lag_cols = [c for c in train_final.columns if "_lag" in c]
    feature_cols = pca_cols + lag_cols

    X_train_arr = train_final[feature_cols].to_numpy(dtype=np.float32)
    y_train_arr = train_final["target"].to_numpy(dtype=np.float32)
    X_val_arr = val_final[feature_cols].to_numpy(dtype=np.float32)
    val_era = val_final["era"].to_numpy(copy=True)
    val_target = val_final["target"].to_numpy(dtype=np.float32, copy=True)

    del train_final, val_final
    gc.collect()

    print("X_train_arr shape:", X_train_arr.shape)
    print("X_val_arr shape:", X_val_arr.shape)
    print("배열 준비 완료 - 이제 모델 학습 셀을 실행하세요.")

except FileNotFoundError:
    print("캐시 파일도 사라졌습니다. Cell 1~8을 처음부터 다시 실행해야 합니다.")

캐시 로드 성공! train_final shape: (229600, 1502)
X_train_arr shape: (229600, 1500)
X_val_arr shape: (260800, 1500)
배열 준비 완료 - 이제 모델 학습 셀을 실행하세요.


In [ ]:
# lag 없이 PCA 500개만 사용 (X_train_arr의 앞 500개 컬럼이 PCA, 나머지 1000개가 lag)
N_PCA = len(pca_cols)  # 500

X_train_small = X_train_arr[:, :N_PCA]
X_val_small = X_val_arr[:, :N_PCA]

RANDOM_STATE = 42

model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.02,
    max_depth=6,
    num_leaves=31,
    max_bin=63,
    n_jobs=1,
    random_state=RANDOM_STATE,
)

model.fit(X_train_small, y_train_arr)
val_prediction = model.predict(X_val_small)
print("학습 완료! (lag 없이 PCA만)")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.455818 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31500
[LightGBM] [Info] Number of data points in the train set: 229600, number of used features: 500
[LightGBM] [Info] Start training from score 0.499310


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


학습 완료! (lag 없이 PCA만)


In [ ]:
def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df = pd.DataFrame({"era": era_arr, "target": target_arr, "prediction": pred_arr})
    def era_corr(sub):
        return np.corrcoef(sub["prediction"].rank(pct=True), sub["target"])[0, 1]
    return df.groupby("era").apply(era_corr)

per_era_corr = numerai_corr_arrays(val_era, val_target, val_prediction)
print("Validation mean correlation:", per_era_corr.mean())
print("Validation std correlation :", per_era_corr.std())
print("Sharpe (mean/std)          :", per_era_corr.mean() / per_era_corr.std())

Validation mean correlation: 0.009804818355725985
Validation std correlation : 0.0513087648413861
Sharpe (mean/std)          : 0.19109441410324757


/tmp/ipykernel_37260/3836315072.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("era").apply(era_corr)


In [ ]:
pd.DataFrame({"era": val_era, "prediction": val_prediction}).to_csv(
    "sihyun_pca500_lag_val_predictions.csv", index=False
)
print("저장 완료: sihyun_pca500_lag_val_predictions.csv")

저장 완료: sihyun_pca500_lag_val_predictions.csv


In [ ]:
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

# 1) 캐시 로드
train_final = pd.read_parquet("train_final_cache.parquet")
val_final = pd.read_parquet("val_final_cache.parquet")
print("캐시 로드 성공! train_final shape:", train_final.shape)

pca_cols = [c for c in train_final.columns if c.startswith("pca_") and "_lag" not in c]
lag_cols = [c for c in train_final.columns if "_lag" in c]

# lag1 컬럼만 골라내기 (lag_cols에는 lag1, lag2가 섞여 있음)
lag1_cols = [c for c in lag_cols if "_lag1" in c]
feature_cols = pca_cols + lag1_cols  # PCA(500) + lag1(500) = 1000개만 사용

X_train_arr = train_final[feature_cols].to_numpy(dtype=np.float32)
y_train_arr = train_final["target"].to_numpy(dtype=np.float32)
X_val_arr = val_final[feature_cols].to_numpy(dtype=np.float32)
val_era = val_final["era"].to_numpy(copy=True)
val_target = val_final["target"].to_numpy(dtype=np.float32, copy=True)

del train_final, val_final
gc.collect()

print("사용 feature 개수 (PCA + lag1):", len(feature_cols))
print("X_train_arr shape:", X_train_arr.shape)

# 2) 학습 (lag1만 포함)
RANDOM_STATE = 42

model_mid = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.02,
    max_depth=6,
    num_leaves=31,
    max_bin=63,
    n_jobs=1,
    random_state=RANDOM_STATE,
)

model_mid.fit(X_train_arr, y_train_arr)
val_prediction_mid = model_mid.predict(X_val_arr)
print("학습 완료! (PCA + lag1)")

캐시 로드 성공! train_final shape: (229600, 1502)
사용 feature 개수 (PCA + lag1): 1000
X_train_arr shape: (229600, 1000)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 13.147742 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 63000
[LightGBM] [Info] Number of data points in the train set: 229600, number of used features: 1000
[LightGBM] [Info] Start training from score 0.499310


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


학습 완료! (PCA + lag1)


위에꺼 다시 점수평가까지 넣어서 돌리기 (=밑에 버전)

In [ ]:
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

# 1) 캐시 로드
train_final = pd.read_parquet("train_final_cache.parquet")
val_final = pd.read_parquet("val_final_cache.parquet")
print("캐시 로드 성공! train_final shape:", train_final.shape)

pca_cols = [c for c in train_final.columns if c.startswith("pca_") and "_lag" not in c]
lag_cols = [c for c in train_final.columns if "_lag" in c]
lag1_cols = [c for c in lag_cols if "_lag1" in c]
feature_cols = pca_cols + lag1_cols  # PCA(500) + lag1(500) = 1000개

X_train_arr = train_final[feature_cols].to_numpy(dtype=np.float32)
y_train_arr = train_final["target"].to_numpy(dtype=np.float32)
X_val_arr = val_final[feature_cols].to_numpy(dtype=np.float32)
val_era = val_final["era"].to_numpy(copy=True)
val_target = val_final["target"].to_numpy(dtype=np.float32, copy=True)

del train_final, val_final
gc.collect()

print("사용 feature 개수 (PCA + lag1):", len(feature_cols))
print("X_train_arr shape:", X_train_arr.shape)

# 2) 학습
RANDOM_STATE = 42

model_mid = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.02,
    max_depth=6,
    num_leaves=31,
    max_bin=63,
    n_jobs=1,
    random_state=RANDOM_STATE,
)

model_mid.fit(X_train_arr, y_train_arr)
val_prediction_mid = model_mid.predict(X_val_arr)
print("학습 완료! (PCA + lag1)")

# 3) 점수 평가
def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df = pd.DataFrame({"era": era_arr, "target": target_arr, "prediction": pred_arr})
    def era_corr(sub):
        return np.corrcoef(sub["prediction"].rank(pct=True), sub["target"])[0, 1]
    return df.groupby("era").apply(era_corr, include_groups=False)

per_era_corr_mid = numerai_corr_arrays(val_era, val_target, val_prediction_mid)

print("[PCA + lag1] Validation mean correlation:", per_era_corr_mid.mean())
print("[PCA + lag1] Validation std correlation :", per_era_corr_mid.std())
print("[PCA + lag1] Sharpe (mean/std)          :", per_era_corr_mid.mean() / per_era_corr_mid.std())

# 4) 저장
pd.DataFrame({"era": val_era, "prediction": val_prediction_mid}).to_csv(
    "sihyun_pca500_lag1_val_predictions.csv", index=False
)
print("저장 완료: sihyun_pca500_lag1_val_predictions.csv")

캐시 로드 성공! train_final shape: (229600, 1502)
사용 feature 개수 (PCA + lag1): 1000
X_train_arr shape: (229600, 1000)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.995784 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 63000
[LightGBM] [Info] Number of data points in the train set: 229600, number of used features: 1000
[LightGBM] [Info] Start training from score 0.499310


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


학습 완료! (PCA + lag1)
[PCA + lag1] Validation mean correlation: 0.009967980346306228
[PCA + lag1] Validation std correlation : 0.053480613404909914
[PCA + lag1] Sharpe (mean/std)          : 0.18638492926094774
저장 완료: sihyun_pca500_lag1_val_predictions.csv


In [ ]:
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

ROW_FRACTION = 0.6  # 터지면 0.4, 0.3으로 낮춰서 재시도

# 1) 캐시 로드
train_final = pd.read_parquet("train_final_cache.parquet")
val_final = pd.read_parquet("val_final_cache.parquet")
print("캐시 로드 성공! train_final shape:", train_final.shape)

# lag1+lag2(1500개 컬럼)를 다 쓰기 위해 행 수를 줄임
train_final = train_final.sample(frac=ROW_FRACTION, random_state=42).reset_index(drop=True)
val_final = val_final.sample(frac=ROW_FRACTION, random_state=42).reset_index(drop=True)
print(f"ROW_FRACTION={ROW_FRACTION} 적용, train_final shape:", train_final.shape)

pca_cols = [c for c in train_final.columns if c.startswith("pca_") and "_lag" not in c]
lag_cols = [c for c in train_final.columns if "_lag" in c]
feature_cols = pca_cols + lag_cols  # PCA(500) + lag1(500) + lag2(500) = 1500개 전부

X_train_arr = train_final[feature_cols].to_numpy(dtype=np.float32)
y_train_arr = train_final["target"].to_numpy(dtype=np.float32)
X_val_arr = val_final[feature_cols].to_numpy(dtype=np.float32)
val_era = val_final["era"].to_numpy(copy=True)
val_target = val_final["target"].to_numpy(dtype=np.float32, copy=True)

del train_final, val_final
gc.collect()

print("사용 feature 개수 (PCA + lag1 + lag2):", len(feature_cols))
print("X_train_arr shape:", X_train_arr.shape)

# 2) 학습
RANDOM_STATE = 42

model_full = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.02,
    max_depth=6,
    num_leaves=31,
    max_bin=63,
    n_jobs=1,
    random_state=RANDOM_STATE,
)

model_full.fit(X_train_arr, y_train_arr)
val_prediction_full = model_full.predict(X_val_arr)
print("학습 완료! (PCA + lag1 + lag2, row 60%)")

# 3) 점수 평가
def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df = pd.DataFrame({"era": era_arr, "target": target_arr, "prediction": pred_arr})
    def era_corr(sub):
        return np.corrcoef(sub["prediction"].rank(pct=True), sub["target"])[0, 1]
    return df.groupby("era").apply(era_corr, include_groups=False)

per_era_corr_full = numerai_corr_arrays(val_era, val_target, val_prediction_full)

print("[PCA + lag1 + lag2] Validation mean correlation:", per_era_corr_full.mean())
print("[PCA + lag1 + lag2] Validation std correlation :", per_era_corr_full.std())
print("[PCA + lag1 + lag2] Sharpe (mean/std)          :", per_era_corr_full.mean() / per_era_corr_full.std())

# 4) 저장
pd.DataFrame({"era": val_era, "prediction": val_prediction_full}).to_csv(
    "sihyun_pca500_lag_both_val_predictions.csv", index=False
)
print("저장 완료: sihyun_pca500_lag_both_val_predictions.csv")

캐시 로드 성공! train_final shape: (229600, 1502)
ROW_FRACTION=0.6 적용, train_final shape: (137760, 1502)
사용 feature 개수 (PCA + lag1 + lag2): 1500
X_train_arr shape: (137760, 1500)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 8.027532 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 94500
[LightGBM] [Info] Number of data points in the train set: 137760, number of used features: 1500
[LightGBM] [Info] Start training from score 0.499837


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


학습 완료! (PCA + lag1 + lag2, row 60%)
[PCA + lag1 + lag2] Validation mean correlation: 0.00910277808417076
[PCA + lag1 + lag2] Validation std correlation : 0.06386816807919277
[PCA + lag1 + lag2] Sharpe (mean/std)          : 0.14252449002269568
저장 완료: sihyun_pca500_lag_both_val_predictions.csv


| 버전 | Feature 수 | 사용 row | Mean Correlation | Sharpe |
|---|---|---|---|---|
| PCA만 | 500 | 100% | 0.0098 | **0.191** |
| PCA + lag1 | 1000 | 100% | 0.00997 | 0.186 |
| PCA + lag1 + lag2 | 1500 | 60% | 0.0091 | 0.143 |

전체 전처리 과정을 순서대로 정리해드릴게요.

## 1. Feature Selection (2748개 → 1374개)

**방법**: 상관계수 기반 selection
- `target_cyrusd_20`(메인 타겟)과 각 feature 간의 **피어슨 상관계수**를 계산
- 상관계수 절대값이 **중앙값 이상인 상위 50%** feature만 선택 (2748개 → 1374개)

**왜 이 방법을 썼나**: 원래 계획은 서윤님처럼 LightGBM importance를 쓰는 거였는데, 그건 데이터를 통째로 메모리에 올려야 계산할 수 있어서 RAM이 감당 안 됐어요. 상관계수는 `sum_x`, `sum_x2`, `sum_xy` 같은 누적합만 있으면 계산되기 때문에, **파일을 청크 단위로 조금씩 흘려보내면서(streaming) 전체 데이터를 다 반영**해서 계산할 수 있었어요.

## 2. Scaling + PCA (1374개 → 500개)

**방법**: `StandardScaler` + `IncrementalPCA`
- 두 개 다 `partial_fit()`을 지원해서, 데이터를 여러 청크로 나눠 여러 번 보여줘도 **전체 데이터 기준으로 정확하게 학습**됨
- 전체 train 데이터를 스트리밍하면서 스케일링 파라미터(평균/표준편차)와 PCA 주성분(500개)을 학습

**결과**: PCA 500개 주성분이 원본 정보의 **91.6%** 를 설명함 (explained variance ratio)

## 3. 최종 학습 데이터 만들기 (샘플링 적용)

- selection/scaling/PCA는 전체 데이터로 정확히 학습했지만, **실제로 이 값들을 적용해서 학습용 데이터를 만드는 단계에서는 메모리 한계 때문에 era당 최대 400종목으로 제한**
- 최종 크기: train 229,600행, validation 260,800행

## 4. Lag Feature 추가 (시장 단위 lag)

- Numerai의 `id`는 종목-era 조합마다 고유해서 **개별 종목을 시간순으로 추적할 수 없음** (공식 문서에 명시된 제약)
- 그래서 "종목별 지난주 값"이 아니라, **era별 PCA 평균값(시장 전체 분위기)을 1주 전(lag1), 2주 전(lag2)으로 shift**해서 모든 종목 행에 붙이는 방식 사용

## 5. 최종 모델 (LightGBM)

세 가지 조합을 비교:

| 조합 | Feature 수 | Sharpe |
|---|---|---|
| PCA만 | 500 | **0.191** (최종 선택) |
| PCA + lag1 | 1000 | 0.186 |
| PCA + lag1 + lag2 | 1500 | 0.143 |

**결론**: lag feature는 이 데이터에서는 성능 개선에 도움이 안 됐고, **PCA-only 버전이 가장 좋은 결과**를 냄.